# 🧪 Lab 7: 고급 패턴 테스트

## 테스트 목표

프로덕션에서 활용할 수 있는 고급 패턴을 검증합니다.

| # | 테스트 | 기대 결과 |
|---|---|---|
| 1 | A/B 라우팅 분포 확인 | `x-ab-group` 헤더가 control/experiment로 분산 |
| 2 | SSE 스트리밍 호출 | `data: {...}` 청크 응답 수신 |
| 3 | 스트리밍 vs 비스트리밍 비교 | 스트리밍이 첫 토큰까지 더 빠름 |

> ℹ️ Content Safety와 PTU Spillover 테스트는 추가 리소스(Content Safety, PTU 배포)가 필요하므로
> 이 노트북에서는 A/B 라우팅과 SSE 스트리밍만 테스트합니다.

### 사전 준비

Lab 7 README의 시나리오 1(A/B 라우팅)과 시나리오 3(SSE 스트리밍) 정책이 적용되어 있어야 합니다.

In [ ]:
import os, json, time
import requests
from dotenv import load_dotenv
from collections import Counter

load_dotenv("../../.env", override=True)

APIM_URL = os.getenv("APIM_URL")
APIM_KEY = os.getenv("APIM_SUBSCRIPTION_KEY")
DEPLOYMENT_NAME = os.getenv("DEPLOYMENT_NAME", "gpt-4.1-nano")
API_VERSION = "2025-04-01-preview"

assert APIM_URL, "❌ APIM_URL이 설정되지 않았습니다."
assert APIM_KEY, "❌ APIM_SUBSCRIPTION_KEY가 설정되지 않았습니다."

BASE_URL = f"{APIM_URL}/openai/deployments/{DEPLOYMENT_NAME}/chat/completions"

print("✅ 환경 설정 완료")
print(f"   Base URL: {BASE_URL}")

---
# 테스트 1: A/B 라우팅 분포 확인

A/B 정책이 적용되었다면, 응답 헤더에 `x-ab-group`이 포함됩니다.
- `control` (90%) — 기존 모델
- `experiment` (10%) — 새 모델

30회 호출하여 분포를 확인합니다.

In [ ]:
print("▶ 테스트 1: A/B 라우팅 분포 확인 (30회 호출)\n")

ab_groups = []
ab_errors = 0

for i in range(1, 31):
    resp = requests.post(
        BASE_URL,
        params={"api-version": API_VERSION},
        headers={
            "Content-Type": "application/json",
            "Ocp-Apim-Subscription-Key": APIM_KEY,
        },
        json={"messages": [{"role": "user", "content": "Hi"}], "max_tokens": 5},
        timeout=30
    )

    group = resp.headers.get("x-ab-group", "none")
    ab_groups.append(group)

    if resp.status_code != 200:
        ab_errors += 1

    if i % 10 == 0:
        print(f"  [{i:2d}/30] 완료...")

    time.sleep(0.3)

counts = Counter(ab_groups)
print(f"\n📊 A/B 분포 결과:")
for group, count in sorted(counts.items()):
    pct = count / len(ab_groups) * 100
    bar = "█" * int(pct / 2)
    print(f"  {group:12s}: {count:2d}회 ({pct:5.1f}%) {bar}")

if "none" in counts and counts["none"] == len(ab_groups):
    print("\n⚠️ x-ab-group 헤더가 없습니다. A/B 정책이 적용되지 않은 것 같습니다.")
    print("   → Lab 7 README의 시나리오 1 정책을 적용하세요.")
else:
    print(f"\n✅ A/B 라우팅 동작 확인! (에러: {ab_errors}건)")

---
# 테스트 2: SSE 스트리밍 호출

`stream: true`로 호출하면 Server-Sent Events로 응답이 청크로 도착합니다.

**관찰 포인트:**
- `data: {...}` 형태로 청크가 순차 도착
- 마지막 청크는 `data: [DONE]`
- 첫 번째 청크까지의 시간 (Time to First Token)

In [ ]:
print("▶ 테스트 2: SSE 스트리밍 호출\n")

start = time.time()
resp_stream = requests.post(
    BASE_URL,
    params={"api-version": API_VERSION},
    headers={
        "Content-Type": "application/json",
        "Ocp-Apim-Subscription-Key": APIM_KEY,
    },
    json={
        "messages": [{"role": "user", "content": "Write a haiku about AI."}],
        "max_tokens": 100,
        "stream": True
    },
    stream=True,
    timeout=30
)

print(f"  HTTP Status: {resp_stream.status_code}")
print(f"  Content-Type: {resp_stream.headers.get('Content-Type', 'N/A')}")
print()

chunks = []
full_content = ""
first_token_time = None

for line in resp_stream.iter_lines(decode_unicode=True):
    if not line:
        continue
    if line.startswith("data: "):
        data_str = line[6:]
        if data_str == "[DONE]":
            print(f"  ← [DONE]")
            break
        try:
            chunk = json.loads(data_str)
            delta = chunk.get("choices", [{}])[0].get("delta", {})
            token = delta.get("content", "")
            if token and first_token_time is None:
                first_token_time = time.time() - start
            full_content += token
            chunks.append(chunk)
            if len(chunks) <= 5:
                print(f"  ← chunk {len(chunks)}: \"{token}\"")
            elif len(chunks) == 6:
                print(f"  ... (이후 청크 생략)")
        except json.JSONDecodeError:
            pass

total_time = time.time() - start
print(f"\n  📊 스트리밍 결과:")
print(f"     총 청크 수: {len(chunks)}")
print(f"     첫 토큰까지: {first_token_time:.2f}s" if first_token_time else "     첫 토큰: N/A")
print(f"     전체 시간: {total_time:.2f}s")
print(f"     전체 응답: {full_content}")

if len(chunks) > 1:
    print(f"\n  ✅ SSE 스트리밍 정상 동작! ({len(chunks)}개 청크 수신)")
else:
    print(f"\n  ⚠️ 청크가 1개 이하입니다. 스트리밍 정책을 확인하세요.")

---
# 테스트 3: 스트리밍 vs 비스트리밍 응답 시간 비교

동일한 프롬프트로 스트리밍/비스트리밍 호출 시간을 비교합니다.

- **비스트리밍**: 전체 응답이 한 번에 (전체 완료까지 대기)
- **스트리밍**: 첫 토큰이 빠르게 도착 (사용자 체감 속도 향상)

In [ ]:
print("▶ 테스트 3: 스트리밍 vs 비스트리밍 비교\n")

prompt = "Explain what an API gateway is in 2 sentences."

# 비스트리밍 호출
start_sync = time.time()
resp_sync = requests.post(
    BASE_URL,
    params={"api-version": API_VERSION},
    headers={"Content-Type": "application/json", "Ocp-Apim-Subscription-Key": APIM_KEY},
    json={"messages": [{"role": "user", "content": prompt}], "max_tokens": 100},
    timeout=30
)
sync_time = time.time() - start_sync

# 스트리밍 호출 — 첫 토큰까지 시간 측정
start_stream = time.time()
resp_str = requests.post(
    BASE_URL,
    params={"api-version": API_VERSION},
    headers={"Content-Type": "application/json", "Ocp-Apim-Subscription-Key": APIM_KEY},
    json={"messages": [{"role": "user", "content": prompt}], "max_tokens": 100, "stream": True},
    stream=True,
    timeout=30
)

stream_first_token = None
for line in resp_str.iter_lines(decode_unicode=True):
    if line and line.startswith("data: ") and line[6:] != "[DONE]":
        try:
            chunk = json.loads(line[6:])
            if chunk.get("choices", [{}])[0].get("delta", {}).get("content"):
                stream_first_token = time.time() - start_stream
                break
        except json.JSONDecodeError:
            pass
# 나머지 소비
for _ in resp_str.iter_lines():
    pass
stream_total = time.time() - start_stream

print(f"  비스트리밍:")
print(f"    전체 응답 시간: {sync_time:.2f}s")
print()
print(f"  스트리밍:")
print(f"    첫 토큰까지:   {stream_first_token:.2f}s" if stream_first_token else "    첫 토큰: N/A")
print(f"    전체 완료:     {stream_total:.2f}s")
print()

if stream_first_token and stream_first_token < sync_time:
    improvement = (1 - stream_first_token / sync_time) * 100
    print(f"  ✅ 스트리밍의 첫 토큰이 {improvement:.0f}% 더 빠릅니다!")
    print(f"     → 사용자 체감 응답 속도가 크게 향상됩니다.")
else:
    print(f"  ℹ️ 이번 테스트에서는 차이가 미미합니다. 긴 응답일수록 차이가 커집니다.")

---
## 결과 요약

In [ ]:
print("═" * 55)
print("  고급 패턴 테스트 요약")
print("═" * 55)
print()

# A/B 결과
ab_ok = "none" not in counts or counts["none"] < len(ab_groups)
print(f"  A/B 라우팅:    {'✅ 동작' if ab_ok else '⚠️ 미적용'}")
if ab_ok:
    for g, c in sorted(counts.items()):
        print(f"                  {g}: {c}회 ({c/len(ab_groups)*100:.0f}%)")

print(f"  SSE 스트리밍:  {'✅ 동작' if len(chunks) > 1 else '⚠️ 확인 필요'}")
if stream_first_token:
    print(f"  첫 토큰 시간:  {stream_first_token:.2f}s (비스트리밍: {sync_time:.2f}s)")
print()
print("─" * 55)
print("\n📌 Content Safety, PTU Spillover는 추가 리소스 배포 후 별도 테스트")